In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import glob

if os.path.exists('notebooks'):
    os.chdir('notebooks')
print("Current Directory:", os.getcwd())

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules or os.path.exists('/kaggle')
IN_LOCAL = not IN_COLAB and not IN_KAGGLE

GITHUB_USERNAME = "rbennum"
REPO_NAME = "titanic-survival"
DATASET_KAGGLE = "yasserh/titanic-dataset"

git_token = None

if IN_COLAB:
    import importlib
    print("🤖 Mode: Google Colab")
    colab_userdata = importlib.import_module('google.colab.userdata')
    colab_drive = importlib.import_module('google.colab.drive')
    colab_drive.mount('/content/drive')
    git_token = colab_userdata.get('GITHUB_TOKEN')
    BASE_DATA_DIR = "../data"
    MODEL_DIR = f"/content/drive/MyDrive/ML_Portfolio/{REPO_NAME}/models"
elif IN_KAGGLE:
    import importlib
    print("🦅 Mode: Kaggle Notebook")
    kaggle_secrets = importlib.import_module('kaggle_secrets')
    user_secrets = kaggle_secrets.UserSecretsClient()
    git_token = user_secrets.get_secret("GITHUB_TOKEN")
    BASE_DATA_DIR = "/kaggle/working/data"
    MODEL_DIR = "/kaggle/working/models"
else:
    print("💻 Mode: Local")
    BASE_DATA_DIR = "../data" 
    MODEL_DIR = "../models"

RAW_DATA_DIR = os.path.join(BASE_DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(BASE_DATA_DIR, "processed")

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

if not IN_LOCAL:
    if git_token:
        if not os.path.exists(f"../.git") and os.path.basename(os.getcwd()) != REPO_NAME:
            print(f"📦 Cloning {REPO_NAME} to Cloud Workspace...")
            REPO_URL = f"https://{GITHUB_USERNAME}:{git_token}@://github.com{GITHUB_USERNAME}/{REPO_NAME}.git"
            !git clone {REPO_URL}

            if os.path.exists(f"./{REPO_NAME}/notebooks"):
                os.chdir(f"./{REPO_NAME}/notebooks")
                print(f"🔄 Shifted cloud workspace to: {os.getcwd()}")
        else:
            print("🔄 Repository already synchronized. Pulling changes...")
            !git pull
    else:
        print("❌ ERROR: GITHUB_TOKEN missing from Secrets manager!")

if not IN_LOCAL:
    print(f"📥 Cloud downloading dataset into: {RAW_DATA_DIR}")
    if IN_COLAB:
        os.environ['KAGGLE_CONFIG_DIR'] = "/content/drive/MyDrive/Kaggle"
    !kaggle datasets download -d {DATASET_KAGGLE} --unzip -p {RAW_DATA_DIR}
    downloaded_csvs = glob.glob(os.path.join(RAW_DATA_DIR, "*.csv"))

    if downloaded_csvs:
        old_file_path = downloaded_csvs[0]
        new_file_path = os.path.join(RAW_DATA_DIR, "dataset.csv")

        if old_file_path != new_file_path:
            os.rename(old_file_path, new_file_path)
            print(f"✨ Successfully renamed '{os.path.basename(old_file_path)}' ➡️ 'dataset.csv'")
        else:
            print("ℹ️ File is already named 'dataset.csv'")
    else:
        print("⚠️ Warning: No CSV files were found in the download directory.")
else:
    print("ℹ️ Local environment active. Drop your data chunks in ../data/raw manually.")

print(f"\n✅ Setup Complete!")
print(f"📂 Raw Data Folder relative target: {RAW_DATA_DIR}")
print(f"📂 Processed Data Folder relative target: {PROCESSED_DATA_DIR}")
print(f"📂 Model Outputs Folder target: {MODEL_DIR}")

Current Directory: /Users/bening/Code/_data_portfolios/titanic-survival/notebooks
💻 Mode: Local
ℹ️ Local environment active. Drop your data chunks in ../data/raw manually.

✅ Setup Complete!
📂 Raw Data Folder relative target: ../data/raw
📂 Processed Data Folder relative target: ../data/processed
📂 Model Outputs Folder target: ../models


# Titanic Survivability

Predict survival on the Titanic and get familiar with ML basics.

## Initialization

In [3]:
import pandas as pd

from ml_utils import utils
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
df = pd.read_csv("../data/raw/dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df['Survived'] = df.pop('Survived')
utils.skim_data(df)

Total duplicate rows: 0
DF shape: (891, 12)


,feature,dtype,null_%,negative_%,zero_%,min,max,n_unique,unique_%,sample_values
0,PassengerId,int64,0.000,0.0,0.0,1,891,891,100.00,"[1, 2, 3, 4, 5]"
1,Pclass,int64,0.000,0.0,0.0,1,3,3,0.34,"[3, 1, 2]"
2,Name,str,0.000,-,-,-,-,891,100.00,"[Braund, Mr. Owen Harris, Cumings, Mrs. John Bradley (Florence Briggs Thayer), Heikkinen, Miss. Laina, Futrelle, Mrs. Jacques Heath (Lily May Peel), Allen, Mr. William Henry]"
3,Sex,str,0.000,-,-,-,-,2,0.22,"[male, female]"
4,Age,float64,19.865,0.0,0.0,0.42,80.0,88,9.88,"[22.0, 38.0, 26.0, 35.0, 54.0]"
5,SibSp,int64,0.000,0.0,68.238,0,8,7,0.79,"[1, 0, 3, 4, 2]"
6,Parch,int64,0.000,0.0,76.094,0,6,7,0.79,"[0, 1, 2, 5, 3]"
7,Ticket,str,0.000,-,-,-,-,681,76.43,"[A/5 21171, PC 17599, STON/O2. 3101282, 113803, 373450]"
8,Fare,float64,0.000,0.0,1.684,0.0,512.3292,248,27.83,"[7.25, 71.2833, 7.925, 53.1, 8.05]"
9,Cabin,str,77.104,-,-,-,-,147,16.50,"[C85, C123, E46, G6, C103]"


## Dataset Split

In [6]:
X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=29
)

In [7]:
utils.skim_data(X_train)

Total duplicate rows: 0
DF shape: (712, 11)


,feature,dtype,null_%,negative_%,zero_%,min,max,n_unique,unique_%,sample_values
0,PassengerId,int64,0.000,0.0,0.0,2,891,712,100.00,"[285, 862, 191, 44, 742]"
1,Pclass,int64,0.000,0.0,0.0,1,3,3,0.42,"[1, 2, 3]"
2,Name,str,0.000,-,-,-,-,712,100.00,"[Smith, Mr. Richard William, Giles, Mr. Frederick Edward, Pinsky, Mrs. (Rosa), Laroche, Miss. Simonne Marie Anne Andree, Cavendish, Mr. Tyrell William]"
3,Sex,str,0.000,-,-,-,-,2,0.28,"[male, female]"
4,Age,float64,19.242,0.0,0.0,0.42,80.0,81,11.38,"[21.0, 32.0, 3.0, 36.0, 28.0]"
5,SibSp,int64,0.000,0.0,68.399,0,8,7,0.98,"[0, 1, 3, 4, 2]"
6,Parch,int64,0.000,0.0,75.983,0,6,7,0.98,"[0, 2, 1, 4, 3]"
7,Ticket,str,0.000,-,-,-,-,571,80.20,"[113056, 28134, 234604, SC/Paris 2123, 19877]"
8,Fare,float64,0.000,0.0,1.826,0.0,512.3292,222,31.18,"[26.0, 11.5, 13.0, 41.5792, 78.85]"
9,Cabin,str,77.528,-,-,-,-,119,16.71,"[A19, C46, B78, C93, C95]"


In [8]:
utils.skim_data(X_test)

Total duplicate rows: 0
DF shape: (179, 11)


,feature,dtype,null_%,negative_%,zero_%,min,max,n_unique,unique_%,sample_values
0,PassengerId,int64,0.000,0.0,0.0,1,890,179,100.00,"[702, 374, 792, 79, 865]"
1,Pclass,int64,0.000,0.0,0.0,1,3,3,1.68,"[1, 2, 3]"
2,Name,str,0.000,-,-,-,-,179,100.00,"[Silverthorne, Mr. Spencer Victor, Ringhini, Mr. Sante, Gaskell, Mr. Alfred, Caldwell, Master. Alden Gates, Gill, Mr. John William]"
3,Sex,str,0.000,-,-,-,-,2,1.12,"[male, female]"
4,Age,float64,22.346,0.0,0.0,0.67,74.0,61,34.08,"[35.0, 22.0, 16.0, 0.83, 24.0]"
5,SibSp,int64,0.000,0.0,67.598,0,8,7,3.91,"[0, 1, 4, 2, 3]"
6,Parch,int64,0.000,0.0,76.536,0,5,5,2.79,"[0, 2, 1, 5, 4]"
7,Ticket,str,0.000,-,-,-,-,165,92.18,"[PC 17475, PC 17760, 239865, 248738, 233866]"
8,Fare,float64,0.000,0.0,1.117,0.0,512.3292,100,55.87,"[26.2875, 135.6333, 26.0, 29.0, 13.0]"
9,Cabin,str,75.419,-,-,-,-,41,22.91,"[E24, E67, C22 C26, B5, A26]"


Based on this first look, the dataset appears to be in good shape. There are no obvious errors like negative values in numerical columns or placeholder text in object-type features. However, a few key points require attention:

*   The `Name` column contains titles and salutations (e.g., Mr., Mrs., Capt.), which could be a valuable source of information.
*   The `Age` and `Cabin` columns have a significant number of null values that will need to be addressed.
*   A very small percentage of data is missing from the `Embarked` column.

To ensure the integrity of the original data throughout the cleaning process, I will split the data into a training set and a test set.

In [12]:
X_train_path = os.path.join(PROCESSED_DATA_DIR, "X_train.csv")
X_test_path = os.path.join(PROCESSED_DATA_DIR, "X_test.csv")
y_train_path = os.path.join(PROCESSED_DATA_DIR, "Y_train.csv")
y_test_path = os.path.join(PROCESSED_DATA_DIR, "Y_test.csv")

X_train.to_csv(X_train_path, index=False)
X_test.to_csv(X_test_path, index=False)
y_train.to_csv(y_train_path, index=False)
y_test.to_csv(y_test_path, index=False)

---

In [10]:
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules or os.path.exists('/kaggle')
IN_CLOUD = IN_COLAB or IN_KAGGLE

if IN_CLOUD:
    print("☁️ Cloud environment detected. Preparing to push changes to GitHub...")
    GIT_EMAIL = "dev.rbennum@gmail.com"
    GIT_NAME = "Bening R."
    COMMIT_MESSAGE = f"{""}"

    !git config --global user.email "{GIT_EMAIL}"
    !git config --global user.name "{GIT_NAME}"

    print("🔄 Staging files...")
    !git add .
    status = !git status --porcelain
    if status:
        print("📝 Committing changes...")
        !git commit -m "{COMMIT_MESSAGE}"
        print("🚀 Pushing to GitHub (Main Branch)...")
        !git push origin main
        print("✨ Success! Your cloud changes are now safely backed up on GitHub.")
    else:
        print("ℹ️ No changes detected in the notebooks. GitHub is already up to date.")
else:
    print("💻 Running locally on MacBook. Use your standard Git terminal or VS Code UI to push your work!")

💻 Running locally on MacBook. Use your standard Git terminal or VS Code UI to push your work!
